# DPR / Transformer Encoder — Updated Change Proposal & Validation

**Reviewed version:** `transformer-encoders-dpr_2.zip` (newer colleague version, supplied 20 Nov 2026)  
**Purpose:** re-assess the previous proposal against the newer code, identify what is still broken or risky, and provide tested replacement patterns.  
**Scope:** code review plus small deterministic regression tests. No repository files are edited by this notebook.

> **Clean-run instruction:** Put this notebook in the repository root, **the same directory as `main.py`**, and run the notebook from that directory. The import-path setup below makes `src/...` imports work in the same manner as the project entry points. Some tests in this notebook are intentionally dependency-light and use synthetic examples; running the full project tests/training additionally requires installing the project's dependencies and dataset/tokenizer access.

## Executive summary

The newer version has fixed several things from the earlier review: the DPR model is now constructed, the DPR loss and trainer are implemented, checkpoint loading is independent for the two encoders, and validation/checkpoint/grid-search scaffolding exists. **The earlier report should not be carried forward unchanged.**

The most important issues still present or introduced are:

1. `requirements.txt` contains Git merge-conflict markers and is not a valid install requirements file.
2. The corpus construction is still not a proper retrieval benchmark: FiQA `main` uses answer strings as candidate passages; SQuAD context strings are wrapped as per-query passages, duplicated, and relevance IDs are synthetic rather than proper corpus qrels.
3. TF-IDF still has multiple runtime bugs; BM25 is still not operational.
4. The legacy recall function is still incorrect for recall@k; precision and MRR are absent from the primary evaluation.
5. The encoder's core math is broadly reasonable and the existing encoder test passes, but attention can produce NaNs on fully masked rows and the full model does not expose attention maps.
6. DPR is trainable now, but it is trained from scratch using a BERT tokenizer only, one randomly sampled positive per query, and in-batch negatives. These choices may limit semantic retrieval, and dataset problems undermine its reported validation quality.
7. `main.py` remains the TF-IDF-only evaluation entry point; DPR training and inference are separate scripts. This is not itself wrong, but it should be made explicit in the project workflow.
8. Vector/checkpoint helpers need stronger input validation and consistency safeguards in the lightweight vector DB path.

**Test status in this environment:** `tests/test_encoder.py` and `tests/test_vector_database.py` pass (**8 tests total**). Full `pytest` collection is blocked by missing `datasets`, `hydra`, and `transformers` packages (five collection errors). TF-IDF/BM25 initialization was directly reproduced to fail on a two-document fixture.

## 0. Repository setup and reproducible context

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "main.py").is_file(), (
    "Run this notebook from the repository root, or place it beside main.py and change cwd accordingly."
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repository root:", REPO_ROOT)
for package in ("torch", "numpy", "scipy", "pytest", "hydra", "transformers", "datasets", "faiss"):
    print(f"{package:14s}", "available" if importlib.util.find_spec(package) else "MISSING")

Repository root: /mnt/data/newer_repo/transformer-encoders-dpr_2
torch          available
numpy          available
scipy          available
pytest         available
hydra          MISSING
transformers   MISSING
datasets       MISSING
faiss          MISSING


## 1. Dependency file contains merge-conflict markers

### A) Current implementation

The supplied `requirements.txt` contains literal lines like:

```text
<<<<<<< HEAD
matplotlib
faiss-cpu
=======
re
>>>>>>> 8a306a23bc0d1109c1037ea3779f077544110031
```

### B) Why it is wrong

These are Git conflict markers, not package declarations. `pip install -r requirements.txt` can fail parsing or treat these as invalid requirement strings. The conflicting section also leaves the intended dependency set unclear. `re` is part of the Python standard library and should not be listed as a pip dependency.

### C) Proposed implementation

A minimal cleaned dependency list should retain the actual runtime/test dependencies used by this project. Pinning versions should be done after selecting and testing a compatible environment; this proposal does not invent exact pins.

### D) Why this is right

A syntactically valid dependency file is necessary for reproducible setup. Explicit dependencies avoid relying on packages incidentally installed on a developer's machine.

### E) Repository change

Replace `requirements.txt` with a clean list; remove every merge marker. Include `matplotlib` if training-history plots are required and `faiss-cpu` only if the FAISS index pipeline is used. Consider separate `requirements-core.txt`, `requirements-train.txt`, and `requirements-faiss.txt` files if you want optional dependencies.

In [2]:
# Proposed cleaned requirements (illustrative, not version-pinned)
clean_requirements = [
    "hydra-core",
    "transformers",
    "torch",
    "datasets",
    "scipy",
    "matplotlib",  # train.py writes plots
    "faiss-cpu",   # optional: build_index.py / FaissVectorStore pipeline
]
assert all(not line.startswith(("<<<<<<<", "=======", ">>>>>>>")) for line in clean_requirements)
assert "re" not in clean_requirements  # stdlib; not installed via pip
print("Clean requirements list:", clean_requirements)

Clean requirements list: ['hydra-core', 'transformers', 'torch', 'datasets', 'scipy', 'matplotlib', 'faiss-cpu']


## 2. Corpus and relevance data model — still fundamentally unsuitable

### A) Current implementation

`src/data_utils/download_retrieval_datasets.py` still loads FiQA with `load_dataset("vibrantlabsai/fiqa", "main", ...)`. `Corpus.__rename_columns()` maps FiQA `ground_truths` to `text` and SQuAD `context` to `text`; for SQuAD it wraps each context as a one-element list. `__assign_ids()` then creates a new synthetic document ID for every answer/context item associated with each query.

### B) Why it is wrong / misleading

- **FiQA:** reference answer text is being treated as a candidate passage corpus. The `main` config is not the independent retrieval corpus plus query relevance labels. A retrieval evaluation over answer strings does not measure retrieval from a real passage collection.
- **SQuAD:** context passages are repeated for many questions. Assigning a new passage ID per query duplicates the same context and can mark copies as irrelevant for other questions, creating artificial false negatives.
- Synthetic IDs are fine as identifiers, but the query-to-document mapping must be based on actual relevance annotations (qrels), not generated as “every answer attached to this query is relevant” unless the dataset definition explicitly says so.

### C) Proposed implementation

Represent the three entities independently: **passages**, **queries**, and **qrels**. Use the actual corpus and relevance files/configuration from a retrieval benchmark such as a properly structured FiQA IR dataset; do not fabricate qrels from reference answers. Deduplicate SQuAD contexts by stable context identity/text and map each question to the shared passage ID.

In [3]:
# Proposed simple schema and validation. This is a contract, not a replacement dataset loader.
passages = [
    {"passage_id": "p1", "text": "A passage about financial risk."},
    {"passage_id": "p2", "text": "A passage about river ecosystems."},
]
queries = [{"query_id": "q1", "query": "financial risk"}, {"query_id": "q2", "query": "river"}]
qrels = {"q1": {"p1": 1}, "q2": {"p2": 1}}

def validate_retrieval_data(passages, queries, qrels):
    pids = [p["passage_id"] for p in passages]
    qids = [q["query_id"] for q in queries]
    assert len(pids) == len(set(pids)) and all(isinstance(pid, str) and pid for pid in pids)
    assert all(isinstance(p["text"], str) and p["text"].strip() for p in passages)
    assert len(qids) == len(set(qids)) and all(isinstance(qid, str) and qid for qid in qids)
    assert all(isinstance(q["query"], str) and q["query"].strip() for q in queries)
    passage_set, query_set = set(pids), set(qids)
    assert set(qrels).issubset(query_set)
    for qid, rels in qrels.items():
        assert rels and set(rels).issubset(passage_set), f"Unknown or empty qrels for {qid}"
        assert all(isinstance(score, (int, float)) and score > 0 for score in rels.values())
    return True
assert validate_retrieval_data(passages, queries, qrels)
print("Schema fixture validated")

Schema fixture validated


### D) Why this is right

- Candidate documents are independent of query/answer annotations.
- Shared passages can be relevant to multiple queries without duplicating the document.
- Qrels are explicit and auditable.
- Evaluation can retrieve from the same corpus for each split while relevance remains split-specific.

### E) Repository change

Rewrite `src/data_utils/download_retrieval_datasets.py` and `src/data_utils/corpus.py` around a corpus/query/qrels schema. For SQuAD, deduplicate identical contexts and build a qrels map from question IDs to context IDs. For FiQA, choose and document a benchmark version that provides a real passage corpus and relevance annotations. Rebuild the cache in a new directory after the schema change; do not reuse old cached processed data.

## 3. TF-IDF — several original bugs remain

### A) Current implementation (from `src/retrievers/tf_idf.py`)

```python
all_docs_text = list(*processed_texts)
...
tf = doc_term_count[term] / sum(doc_term_count.values())
...
if term in self.term_to_index():
    term_idx = self.term_to_index[term]
```

### B) Why it is wrong

1. `list(*processed_texts)` is invalid when the iterator yields multiple sets: `list` receives multiple positional arguments. The failure was reproduced for two documents.
2. A term is missing from many document term-count dictionaries; direct `doc_term_count[term]` indexing raises `KeyError`.
3. `self.term_to_index` is a dictionary, but it is called like a function (`self.term_to_index()`), causing `TypeError` in query vectorization.
4. Preprocessing uses `re.sub(str([character]), ...)`, which constructs an unintended regex from a list representation. Punctuation removal is inconsistent.
5. Empty documents can cause division by zero during TF normalization.

### C) Proposed implementation

The following reference implementation uses a clean tokenizer, safe counts, and the standard smoothed IDF variant. This is a clear, small implementation suitable for testing; choose the exact TF/IDF convention to match the assignment/report.

In [4]:
import re, math
from collections import Counter
import numpy as np

def clean_tokens(text):
    # Normalize punctuation to spaces rather than concatenating neighboring words.
    return re.findall(r"\b\w+\b", text.casefold())

def tfidf_fit_transform(texts):
    if not texts or any(not isinstance(t, str) for t in texts):
        raise ValueError("texts must be a nonempty list of strings")
    tokenized = [clean_tokens(t) for t in texts]
    vocab = sorted({token for row in tokenized for token in row})
    if not vocab:
        raise ValueError("Corpus has no tokens")
    term_to_idx = {term: i for i, term in enumerate(vocab)}
    n_docs = len(texts)
    df = Counter(term for row in tokenized for term in set(row))
    idf = {term: math.log((n_docs + 1) / (df[term] + 1)) + 1.0 for term in vocab}
    matrix = np.zeros((n_docs, len(vocab)), dtype=np.float64)
    for i, row in enumerate(tokenized):
        counts = Counter(row)
        total = len(row)
        if total:
            for term, count in counts.items():
                matrix[i, term_to_idx[term]] = (count / total) * idf[term]
    return vocab, term_to_idx, idf, matrix

def tfidf_query_vector(query, vocab, idf):
    counts = Counter(clean_tokens(query))
    total = sum(counts.values()) or 1
    return np.array([(counts[t] / total) * idf.get(t, 0.0) for t in vocab], dtype=np.float64)

### D) Why it is right

- Document frequency is counted once per document.
- Missing terms naturally contribute zero rather than raising errors.
- Unknown query terms are ignored safely.
- Empty documents remain zero rows.
- Punctuation handling is deterministic.

This is a tested baseline implementation; sparse matrices can be introduced later for efficiency without changing the mathematical contract.

### E) Repository change

Replace `GetDocumentFrequencies`, `GetDocumentsTFIDFVectors`, `QueryToVector`, and `PreprocessText` in `src/retrievers/tf_idf.py`. Add unit tests for multi-document corpora, terms absent from documents, unseen query terms, punctuation, empty strings, and score ranking. Keep the chosen TF/IDF convention documented.

In [5]:
# TF-IDF reference tests
texts = ["alpha alpha beta", "beta gamma", "punctuation, works!", ""]
vocab, mapping, idf, tfidf = tfidf_fit_transform(texts)
assert tfidf.shape == (4, len(vocab)) and np.isfinite(tfidf).all()
assert mapping["alpha"] in range(len(vocab))
q = tfidf_query_vector("alpha missing", vocab, idf)
assert np.isfinite(q).all() and q[mapping["alpha"]] > 0
assert q[mapping["beta"]] == 0
assert clean_tokens("Hello, world! A-B") == ["hello", "world", "a", "b"]
print("TF-IDF reference tests passed")

TF-IDF reference tests passed


## 4. BM25 — still uninitialized and still inherits broken TF-IDF

### A) Current implementation

`BM25.__init__()` calls `TFIDF.__init__()` and then computes BM25 IDF. `GetDocumentsBM25Vectors()` exists, but it is never called, so `self.bm25_matrix` is not created. `CalculateScores()` nevertheless uses `self.bm25_matrix`. The matrix builder also indexes `doc_term_count[term]` for terms absent from a document.

### B) Why it is wrong

BM25 scoring cannot run with an uninitialized matrix. In addition, inherited TF-IDF initialization currently fails on multi-document input because of the document-frequency bug. BM25 should not depend on a successfully constructed TF-IDF document matrix at all.

### C) Proposed implementation

Use a direct BM25 score calculation with term frequencies, document lengths, document frequency, and query term counts. The conventional nonnegative IDF variant below avoids negative IDF values for very frequent terms.

In [6]:
def bm25_scores(query, documents, k1=1.5, b=0.75):
    if not documents:
        raise ValueError("documents must be nonempty")
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("Require k1 > 0 and b in [0, 1]")
    tokenized = [clean_tokens(doc) for doc in documents]
    q_terms = set(clean_tokens(query))
    N = len(tokenized)
    lengths = np.array([len(doc) for doc in tokenized], dtype=float)
    avgdl = lengths.mean() if N else 0.0
    df = Counter(term for doc in tokenized for term in set(doc))
    scores = np.zeros(N, dtype=float)
    if avgdl == 0:
        return scores
    for i, tokens in enumerate(tokenized):
        tf = Counter(tokens)
        for term in q_terms:
            freq = tf.get(term, 0)
            if freq == 0:
                continue
            idf = math.log(1 + (N - df[term] + 0.5) / (df[term] + 0.5))
            length_norm = k1 * (1 - b + b * lengths[i] / avgdl)
            scores[i] += idf * (freq * (k1 + 1)) / (freq + length_norm)
    return scores

### D) Why it is right

- BM25 is independently computed; it does not depend on TF-IDF initialization.
- Missing term frequencies safely evaluate to zero.
- The average document length is calculated once.
- Parameters are validated and empty-token corpora return zero scores rather than dividing by zero.
- Scores can be checked against hand-calculated toy cases.

### E) Repository change

Refactor `src/retrievers/bm25.py` so its constructor builds or computes BM25 scoring state. Remove its unnecessary dependency on TF-IDF matrix initialization (or share only the tokenizer/document wrapper). Add tests that instantiate BM25, score multi-document corpora, query absent terms, and confirm relevant documents rank above unrelated ones.

In [7]:
bm_docs = ["alpha alpha beta", "beta gamma", "river water"]
scores = bm25_scores("alpha", bm_docs)
assert scores.shape == (3,) and np.isfinite(scores).all()
assert scores[0] > scores[1] and scores[1] == 0 and scores[2] == 0
assert np.all(bm25_scores("unknown", bm_docs) == 0)
assert np.all(bm25_scores("anything", ["", " "]) == 0)
print("BM25 reference tests passed; scores:", scores)

BM25 reference tests passed; scores: [1.28332799 0.         0.        ]


## 5. Evaluation metrics — legacy recall is precision-like at small k

### A) Current implementation

`src/evaluation/metrics.py` currently has:

```python
def calculateRecall(relevancies, query):
    return sum(relevancies) / min(len(relevancies), query.getNumberOfRelevantDocuments())
```
A separate `mean_recall()` used by the DPR training script divides by the total number of known relevant passages, but the legacy `Evaluate()` path still uses the incorrect function.

### B) Why it is wrong

Recall@k is **relevant retrieved / all relevant in the ground truth**. When the number retrieved is less than the number relevant, dividing by `k` instead makes the score precision@k. Example: 2 relevant hits among 5 retrieved, with 10 relevant passages overall, is recall 0.2—not 0.4.

The main evaluation still only reports recall. Assignment metrics also include Precision@k and MRR.

### C) Proposed implementation

In [8]:
def precision_at_k(flags, k):
    if k < 1: raise ValueError("k must be positive")
    top = flags[:k]
    return sum(bool(x) for x in top) / max(len(top), 1)

def recall_at_k(flags, total_relevant, k):
    if k < 1: raise ValueError("k must be positive")
    if total_relevant < 1: raise ValueError("total_relevant must be positive")
    return sum(bool(x) for x in flags[:k]) / total_relevant

def reciprocal_rank(flags):
    for rank, hit in enumerate(flags, start=1):
        if hit: return 1.0 / rank
    return 0.0

def mean_metrics(results, total_relevant_per_query, k):
    assert len(results) == len(total_relevant_per_query) and len(results) > 0
    return {
        f"precision@{k}": np.mean([precision_at_k(r, k) for r in results]),
        f"recall@{k}": np.mean([recall_at_k(r, n, k) for r, n in zip(results, total_relevant_per_query)]),
        "mrr": np.mean([reciprocal_rank(r) for r in results]),
    }

### D) Why it is right

Each metric now uses its standard denominator/definition. Recall can be below precision at the same cutoff, MRR rewards early first relevant results, and empty/invalid denominators are handled explicitly.

### E) Repository change

Replace `calculateRecall` and expand `Evaluate` in `src/evaluation/metrics.py`. Ensure both TF-IDF, BM25 and DPR evaluation use the same query list, corpus, qrels, and metric implementation. Report `k` in the result names and avoid tuning on test data.

In [9]:
# Metric sanity check: 2 hits in top 5, 10 relevant overall => recall@5 = .2, not .4
flags = [True, False, True, False, False]
assert precision_at_k(flags, 5) == 2/5
assert recall_at_k(flags, 10, 5) == 2/10
assert reciprocal_rank(flags) == 1.0
assert reciprocal_rank([False, True]) == 0.5
assert reciprocal_rank([False, False]) == 0.0
print("Metric tests passed")

Metric tests passed


## 6. Transformer encoder — keep the architecture, harden edge cases

### A) Current implementation

The newer `src/retrievers/models/encoder.py` retains the same core encoder as the earlier version: scaled dot-product attention, multi-head projections, sinusoidal positions, ReLU feed-forward layers, residual connections, layer normalization, and mean/max/CLS pooling. The test suite validates shapes, padding behavior, and gradients.

Attention masks use `-inf` for blocked positions before softmax.

### B) Why it needs improvement

The core math looks conventional and the test passes, so a wholesale rewrite is not justified. But a fully masked attention row causes softmax over all `-inf`, yielding NaNs. This is an edge-case robustness issue. Also, `TransformerEncoder` does not collect attention weights across layers, despite the assignment's attention-analysis requirement. Embedding scaling/dropout are design choices to experiment with—not confirmed correctness bugs.

### C) Proposed implementation

A robust attention helper should zero the weights of fully masked rows rather than produce NaNs. This function preserves normal softmax behavior for rows with at least one allowed key and returns zero attention for all-blocked rows.

In [10]:
import torch
import torch.nn.functional as F
from math import sqrt

def safe_scaled_dot_attention(q, k, v, mask=None):
    scores = (q @ k.transpose(-2, -1)) / sqrt(q.shape[-1])
    if mask is None:
        weights = torch.softmax(scores, dim=-1)
    else:
        mask = mask.to(device=scores.device, dtype=torch.bool)
        masked = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(masked, dim=-1) * mask
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(torch.finfo(weights.dtype).tiny)
    return weights @ v, weights

### D) Why it is right

- Fully masked rows yield finite zero weights/output.
- Valid rows still sum to one across permitted keys.
- Blocked positions have exactly zero weight.
- The test below exercises both normal and fully blocked rows.

### E) Repository change

Update `ScaledDotProductAttention.forward` in `src/retrievers/models/encoder.py`; add tests for all-masked rows, mixed valid/blocked keys, and padding invariance. Extend `EncoderBlock`/`TransformerEncoder` with an opt-in `return_attention` collection that returns per-layer/per-head attention tensors. Keep the existing default output signature stable if possible.

In [11]:
# Attention regression checks (using B x H x L x D shapes)
q = torch.tensor([[[[1., 0.], [0., 1.]]]])
k = q.clone(); v = torch.tensor([[[[2., 0.], [0., 6.]]]])
valid_mask = torch.tensor([[[[True, False], [True, True]]]])
out, weights = safe_scaled_dot_attention(q, k, v, valid_mask)
assert torch.isfinite(out).all() and torch.isfinite(weights).all()
assert torch.allclose(weights[0,0,0], torch.tensor([1., 0.]))
all_blocked = torch.zeros_like(valid_mask, dtype=torch.bool)
out0, w0 = safe_scaled_dot_attention(q, k, v, all_blocked)
assert torch.equal(w0, torch.zeros_like(w0)) and torch.equal(out0, torch.zeros_like(out0))
print("Safe attention tests passed")

Safe attention tests passed


## 7. DPR loss — implemented, but validate positive-mask contract

### A) Current implementation

`src/training/loss.py` now implements in-batch contrastive loss. With no `positive_mask`, it uses cross-entropy against diagonal targets. With a mask, it uses log-softmax and log-sum-exp over positives, allowing multiple positives per query.

### B) Why it still needs improvement

The implementation assumes the mask has the correct shape/type and every query has at least one positive. An empty-positive row makes its log-sum-exp `-inf`, which yields infinite loss. Shape mismatch errors are left to PyTorch and can be confusing. The mathematical multi-positive formulation itself is appropriate.

### C) Proposed implementation

In [12]:
def validated_dpr_loss(query_vecs, passage_vecs, positive_mask=None):
    if query_vecs.ndim != 2 or passage_vecs.ndim != 2:
        raise ValueError("Query and passage embeddings must be 2D")
    if query_vecs.shape[1] != passage_vecs.shape[1]:
        raise ValueError("Query and passage embedding dimensions must match")
    if not torch.isfinite(query_vecs).all() or not torch.isfinite(passage_vecs).all():
        raise ValueError("Embeddings must be finite")
    scores = query_vecs @ passage_vecs.T
    if positive_mask is None:
        if scores.shape[0] != scores.shape[1]:
            raise ValueError("Diagonal-target loss requires equal query and passage batch sizes")
        return F.cross_entropy(scores, torch.arange(scores.shape[0], device=scores.device))
    if positive_mask.shape != scores.shape or positive_mask.dtype != torch.bool:
        raise ValueError("positive_mask must be bool with shape (n_queries, n_passages)")
    if not positive_mask.any(dim=1).all():
        raise ValueError("Every query must have at least one positive passage")
    log_probs = F.log_softmax(scores, dim=1)
    return -torch.logsumexp(log_probs.masked_fill(~positive_mask, float("-inf")), dim=1).mean()

### D) Why it is right

The contract is checked before the loss is calculated. Multi-positive loss still rewards the probability mass assigned to any known positive passage, while invalid rows are rejected rather than silently producing non-finite training values.

### E) Repository change

Add input validation in `src/training/loss.py` and tests for diagonal positives, multiple positives, no-positive rows, wrong mask shape/dtype, unequal embedding widths, and non-finite values.

In [13]:
# Loss tests: finite diagonal loss, valid multi-positive loss, invalid positive mask rejected
qv = torch.tensor([[1., 0.], [0., 1.]])
dv = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
mask = torch.tensor([[True, False, True], [False, True, True]])
loss_multi = validated_dpr_loss(qv, dv, mask)
assert torch.isfinite(loss_multi) and loss_multi >= 0
loss_single = validated_dpr_loss(qv, dv[:2])
assert torch.isfinite(loss_single)
try:
    validated_dpr_loss(qv, dv, torch.tensor([[False, False, False], [False, True, False]]))
except ValueError as e:
    assert "positive" in str(e)
else:
    raise AssertionError("Expected invalid positive-mask row to be rejected")
print("DPR loss tests passed; multi-positive loss:", float(loss_multi))

DPR loss tests passed; multi-positive loss: 0.16884762048721313


## 8. Trainer — works now, but negative sampling and data assumptions need review

### A) Current implementation

The newer `src/training/trainer.py` is implemented. It:

- samples one positive passage for each query per batch;
- uses other batch passages as in-batch negatives;
- constructs a multi-positive mask based on positive passages known for each query string;
- optimizes both encoders with AdamW;
- reports loss, positive/negative scores, gradient norms and epoch duration;
- prepares fixed validation batches and computes validation loss.

### B) Why this may still limit quality

- There is no explicit hard-negative mining. Negatives are just passages selected as positives for other queries in the same batch.
- A passage relevant to another query may be treated as a negative unless that relevance is represented in the current `positives_by_query` mapping. The current data schema is answer/context-per-query, so this is especially problematic.
- Only one positive is sampled per query per batch. Multi-positive masking protects other selected known positives, but the training signal depends on positive sampling and the correctness of the qrels.
- Batch size controls negative count. Larger batches may help but also consume memory and can include false negatives.
- The trainer has no learning-rate scheduler or gradient clipping (potential improvements to experiment with, not demonstrated bugs).

### C) Proposed implementation / training strategy

Keep the working trainer as the baseline, but define negatives from explicit qrels, maintain a query-to-positive-ID map independent of text, and make sampling reproducible. A future extension can add mined hard negatives after a stable baseline is established. Track validation Recall@k, MRR and loss, and choose checkpoints only using validation data.

In [14]:
# Test the positive-mask contract using passage IDs, not only passage text.
def build_positive_mask(query_ids, candidate_passage_ids, qrels):
    return torch.tensor([
        [pid in qrels.get(qid, set()) for pid in candidate_passage_ids]
        for qid in query_ids
    ], dtype=torch.bool)
qrels_by_id = {"q1": {"p1", "p3"}, "q2": {"p2"}}
positive_mask_test = build_positive_mask(["q1", "q2"], ["p1", "p2", "p3"], qrels_by_id)
assert positive_mask_test.tolist() == [[True, False, True], [False, True, False]]
assert positive_mask_test.any(dim=1).all()
print("ID-based positive-mask example passed")

ID-based positive-mask example passed


### D) Why it is right

Explicit qrels and passage IDs avoid relying on passage text equality and allow a passage to be positive for several queries. This makes the in-batch mask auditable. Hard negatives should be added only after the baseline can be trained and evaluated reliably.

### E) Repository change

Refactor `src/training/trainer.py` to consume query IDs, positive passage IDs, and passage lookup data from the corrected dataset layer. Add a test where one passage is relevant to two different queries, and assert it is never marked negative for either query. Preserve the current trainer as a reproducible baseline while introducing new sampling options incrementally.

## 9. Vector database and encoder batching — defensive validation

### A) Current implementation

`VectorDatabase._check()` validates only array dimensionality and embedding width. `insert()` validates ID count and uniqueness within the inserted batch, but does not validate ID type/nonempty strings or vector finiteness. `FlatIndex.search()` clamps `k` to the number of stored rows, but does not reject `k <= 0`. `encode_token_ids()` changes the encoder to eval mode and restores training mode after inference, but restoration is not protected by `finally`.

### B) Why it needs improvement

NaN/Inf vectors can silently contaminate similarity scores. Invalid k values can fail in `argpartition`. Empty token batches cause `torch.cat([])` failures. If encoding throws an exception, the encoder may remain in eval mode, changing later training behavior.

### C) Proposed implementation

In [15]:
def checked_vectors(vectors, dim):
    array = np.asarray(vectors, dtype=np.float32)
    if array.ndim != 2 or array.shape[1] != dim:
        raise ValueError(f"Expected (N, {dim}) vectors")
    if not np.isfinite(array).all():
        raise ValueError("Embedding vectors must be finite")
    return array

def validated_topk(scores, k):
    if k < 1: raise ValueError("k must be positive")
    if scores.ndim != 2 or scores.shape[1] == 0:
        raise ValueError("Scores must have shape (queries, nonempty corpus)")
    k = min(k, scores.shape[1])
    idx = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return idx, np.take_along_axis(scores, idx, axis=1)

def safe_encode_batches(encoder, token_ids, batch_size=64):
    if batch_size < 1: raise ValueError("batch_size must be positive")
    if len(token_ids) == 0: raise ValueError("token_ids must contain at least one row")
    was_training = encoder.training
    encoder.eval()
    try:
        with torch.no_grad():
            chunks = [encoder(token_ids[i:i+batch_size]) for i in range(0, len(token_ids), batch_size)]
            return torch.cat(chunks)
    finally:
        encoder.train(was_training)

### D) Why it is right

It fails early on invalid input, guarantees finite embeddings, defines behavior for `k` larger than the corpus, rejects nonpositive k and empty batches, and restores encoder mode even if encoding raises.

### E) Repository change

Update `src/retrievers/vector_database.py` with finite-value/ID checks, `k` validation, empty-batch checks, and try/finally mode restoration. Add tests for malformed shape, NaN/Inf, invalid k, empty corpus/batch, and encoder mode restoration after both success and injected failure. The newer FAISS store already includes more validation; align the two store implementations or document why both exist.

In [16]:
# Vector validation tests
assert checked_vectors([[1, 2], [3, 4]], 2).dtype == np.float32
for bad in ([[1, float("nan")]], [[1, float("inf")]], [[1, 2, 3]]):
    try: checked_vectors(bad, 2)
    except ValueError: pass
    else: raise AssertionError("bad vectors should be rejected")
score_matrix = np.array([[0.1, 0.9, 0.4], [0.8, 0.2, 0.3]])
idx, ordered_scores = validated_topk(score_matrix, 2)
assert idx.tolist() == [[1, 2], [0, 2]] and ordered_scores.shape == (2, 2)
for kbad in [0, -1]:
    try: validated_topk(score_matrix, kbad)
    except ValueError: pass
    else: raise AssertionError("invalid k should be rejected")
print("Vector/top-k validation tests passed")

Vector/top-k validation tests passed


## 10. Checkpoints and persisted indexes — improved, with consistency checks still important

### A) Current implementation

The newer repository has `src/retrievers/checkpoint.py`, `train.py`, `build_index.py`, and `search.py`. The `best` checkpoint includes encoder state files and metadata containing epoch, validation recall, top-k, query count, and configuration. `load_checkpoint()` loads the config and paths, hashes the two encoder files, and `search.py` compares those hashes with the index metadata. The separate FAISS store checks dimension, metric, and document count when loading.

### B) Why it is still worth improving

This is a significant improvement over the earlier version; checkpoint loading is no longer fundamentally absent. Remaining risks: the root training output encoder files do not have the `best/metadata.json` expected by the checkpoint loader; users can mistakenly load an epoch/root path. The lightweight `VectorDatabase` path stores only generic encoder metadata and has no checkpoint fingerprint, so it can be paired with mismatched encoders. Index rebuilds are required whenever the document encoder changes.

### C) Proposed implementation

Use a single documented artifact layout. Store both encoder weights, tokenizer identifier/revision, encoder config, dataset/corpus version, pooling/max-length settings, and hashes in a manifest. Make load routines reject missing manifest fields and validate that index metadata matches the model manifest.

### D) Why it is right

Dense passage vectors are model-dependent: a vector index is valid only for the passage encoder that produced it. Query and passage encoders must also be the trained compatible pair. Metadata and hashes make incompatible artifact combinations fail loudly rather than returning silently poor rankings.

### E) Repository change

Standardize checkpoint creation/loading in `src/retrievers/checkpoint.py` and `src/retrievers/vector_database.py` / `src/retrievers/vector_store.py`. Document that `outputs/.../best` is the supported inference checkpoint. Add a test that deliberately mismatches one encoder checkpoint and confirms search refuses to use the index.

## 11. Retrieval quality: training-from-scratch is a limitation, not a confirmed bug

### A) Current implementation

The configured tokenizer is `bert-base-uncased`, but `DPR.__load_models()` instantiates the custom `TransformerEncoder` with randomly initialized token embeddings and Transformer weights. The tokenizer supplies token IDs only; it does not supply pretrained model weights. Config defaults include a six-layer, 128-dimensional custom encoder and CLS pooling. Training uses a dual encoder, in-batch contrastive learning, and randomly sampled positive passages.

### B) Why results may be weak

A random-initialized model must learn both general language structure and the retrieval relationship from the task data. A small/medium retrieval training set may not provide enough supervision. BERT's tokenizer alone does not give BERT semantic representations. The current corpus/qrels issues mean validation recall is not a reliable measure of the intended retrieval task. Random positive sampling and in-batch negatives may further limit signal.

This does **not** mean the custom encoder should be replaced: implementing a Transformer is an assignment objective. It does mean a stronger pretrained retrieval model can be used as a comparison/upper-bound experiment if permitted, while the custom model remains the primary required model.

### C) Proposed experiments

1. Fix the corpus and qrels first.
2. Establish a reproducible small overfit test: the model should fit a tiny fixed set of query–positive pairs (sanity check, not generalization).
3. Run a custom encoder baseline and tune pooling, max length, learning rate, batch size, epochs, and dropout on validation only.
4. Compare against TF-IDF and BM25 on identical qrels.
5. If permitted, add a pretrained dual-encoder or pretrained BERT encoder baseline to isolate the performance gap attributable to representation initialization.
6. Add hard-negative mining only after the baseline training loop and qrels are trustworthy.

### D) Why it is right

The experiment ladder separates infrastructure correctness, ability to overfit, generalization, and the effect of pretraining. It avoids changing several variables at once and prevents test-set tuning.

### E) Repository change

Update `config/model/encoder.yaml`, `config/training/dpr.yaml`, the training experiment script, and experiment reporting—not necessarily the architecture itself. Record the model initialization, tokenizer, dataset/corpus version, pooling, maximum length, batch sizes, negative strategy, random seed, and checkpoint selection metric for every run. Use validation metrics for model selection and report test metrics once.

## 12. Entry points and experiment flow

### A) Current implementation

- `main.py` loads `Corpus`, builds TF-IDF, and evaluates train/validation/test recall.
- `train.py` runs DPR training when invoked with `retriever=dpr` and writes training history/checkpoints.
- `grid_search.py` performs sequential runs across pooling, learning-rate, and max-length combinations.
- `build_index.py` and `search.py` are a separate FAISS inference flow.

### B) Why it may confuse users

The project has multiple paths with different assumptions. Running `main.py retriever=dpr` does not train or evaluate DPR; the main script is still a TF-IDF script. `train.py` is the training entry point, and search/index scripts are a separate flow. Also, default experiment configuration uses `tf_idf`, while DPR training requires overrides.

### C) Proposed implementation

Keep entry points separate if intentional, but make command names and README workflows explicit. Add a dedicated evaluation entry point that loads a saved DPR checkpoint, builds or loads the corpus index, runs queries against the correct qrels, and reports Precision@k, Recall@k, and MRR. Avoid implying `main.py` supports every configured retriever.

### D) Why it is right

Clear workflows reduce accidental evaluation of the wrong model and make baseline comparisons reproducible.

### E) Repository change

Update `README.md`, `main.py` help/config, and add `evaluate.py` (or equivalent) for lexical and dense retrieval. Use the same dataset split/qrels/candidate corpus across models. Add a smoke test for each documented command.

## 13. Validation performed on the supplied newer repository

### Results

**Passed:**
- `python -m compileall -q .` (syntax compilation)
- `python -m pytest -q tests/test_encoder.py` → 1 passed
- `python -m pytest -q tests/test_vector_database.py tests/test_encoder.py` → 8 passed
- Two-document TF-IDF and BM25 construction → both reproduced `TypeError: list expected at most 1 argument, got 2` due to `list(*processed_texts)`.

**Blocked:** full `pytest` collection. It stopped with five collection errors because this runtime lacks `datasets`, `hydra`, and `transformers`. The tests for DPR, training, FAISS, grid search, and real-FiQA overfit therefore were not executed here. `faiss` is also missing. No full training or benchmark score was produced.

The following cells re-run selected checks. They are intended to give a clear record when this notebook is run in the project environment.

In [17]:
# Re-run the encoder and vector database tests from this repository.
# This cell requires pytest and torch, both present in the review environment.
encoder_test = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_encoder.py"],
                             cwd=REPO_ROOT, capture_output=True, text=True)
print(encoder_test.stdout)
assert encoder_test.returncode == 0, encoder_test.stderr
vector_test = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_vector_database.py"],
                             cwd=REPO_ROOT, capture_output=True, text=True)
print(vector_test.stdout)
assert vector_test.returncode == 0, vector_test.stderr

.                                                                        [100%]
1 passed in 1.10s



.......                                                                  [100%]
7 passed in 1.00s



In [18]:
# Reproduce the still-broken lexical retrieval initialization without optional dependencies.
# Expected outcome: both initializations fail at TFIDF document-frequency collection.
from src.data_utils.document import Document
from src.retrievers.tf_idf import TFIDF
from src.retrievers.bm25 import BM25
fixture = [{"text": ["alpha beta"], "document_id": ["d1"]},
           {"text": ["beta gamma"], "document_id": ["d2"]}]
observed = {}
for cls in (TFIDF, BM25):
    try:
        cls(documents=fixture)
    except Exception as exc:
        observed[cls.__name__] = f"{type(exc).__name__}: {exc}"
    else:
        observed[cls.__name__] = "unexpectedly initialized"
print(observed)
assert all("TypeError" in observed[name] for name in ("TFIDF", "BM25"))

{'TFIDF': 'TypeError: list expected at most 1 argument, got 2', 'BM25': 'TypeError: list expected at most 1 argument, got 2'}


## 14. Proposed-fix validation summary

In [19]:
checks = {
    "corpus/query/qrels schema validation": True,
    "TF-IDF unknown-term and empty-document handling": True,
    "BM25 multi-document scoring and unknown terms": True,
    "Recall/precision/MRR definitions on toy case": True,
    "fully-masked attention finite zero output": True,
    "DPR loss positive-mask validation": True,
    "ID-based positive mask construction": True,
    "vector shape/finiteness/k validation": True,
}
# These flags are confirmed by the assert-based cells above in this notebook run.
assert all(checks.values())
print(f"{sum(checks.values())}/{len(checks)} proposal checks passed")

8/8 proposal checks passed


## 15. Change checklist for the repository

| Priority | File(s) | Proposed change | Validation required |
|---|---|---|---|
| P0 | `requirements.txt` | Remove merge-conflict markers; define optional dependencies clearly | Fresh environment `pip install -r ...` |
| P0 | `src/data_utils/download_retrieval_datasets.py`, `corpus.py` | Load actual retrieval corpus, queries, and qrels; deduplicate passages; remove answer-as-corpus shortcut | Schema and ID/qrels tests; dataset smoke load |
| P0 | `src/retrievers/tf_idf.py` | Fix document-frequency collection, safe TF, query dictionary access, tokenizer | Multi-doc/empty/unseen-term tests |
| P0 | `src/retrievers/bm25.py` | Initialize/use BM25 scoring state independently; fix missing terms and validation | Multi-doc ranking tests |
| P0 | `src/evaluation/metrics.py` | Correct Recall@k denominator; add Precision@k and MRR | Hand-calculated fixture; shared evaluation tests |
| P1 | `src/retrievers/models/encoder.py` | Fully-masked attention robustness; attention-map return option; validate configuration | Mask, finite-output, shape and attention-map tests |
| P1 | `src/training/loss.py` | Validate embedding/mask inputs and nonempty positives | Multi-positive and invalid-mask tests |
| P1 | `src/training/trainer.py`, `corpus.py` | Train from explicit qrels IDs; prevent known false negatives; evaluate sampling choices | Shared-positive-passage batch test; small overfit test |
| P1 | `src/retrievers/vector_database.py` | Validate finite vectors/IDs/k/empty inputs; restore model mode in `finally` | Bad-input and injected-failure tests |
| P1 | `checkpoint.py`, vector DB/store | Standardize manifest/checkpoint/index linkage | Mismatched checkpoint/index test |
| P2 | `train.py`, `grid_search.py`, `config/` | Add controlled experiments; document custom-from-scratch limitations and model metadata | Repeatable run and validation-only selection |
| P2 | `main.py`, new evaluation entry point, `README.md` | Explicit workflows for TF-IDF, BM25, DPR train/evaluate/index/search | Command smoke tests |


## 16. Recommended order of work

1. **Repair requirements and dataset semantics.** Don't train on the current answer/context-derived “corpus” and call it benchmark retrieval.
2. **Fix TF-IDF and BM25.** Confirm trustworthy baselines first.
3. **Fix evaluation.** All methods should use the same qrels and metrics.
4. **Harden the encoder and loss.** Keep the existing architecture unless tests reveal a core mathematical issue.
5. **Run the small-batch overfit test.** Confirm gradients and loss reduction on a fixed batch.
6. **Train DPR with proper corpus and qrels.** Track validation metrics and save the best checkpoint.
7. **Evaluate once on test.** Compare lexical baselines, custom DPR, and optional pretrained reference model.

## 17. Bottom line

The newer version is materially better than the previous version: DPR construction, loss, trainer, checkpointing, validation, and grid-search scaffolding are now present. The prior assessment that DPR/training were wholly unimplemented is outdated.

**Still broken/confirmed:** requirements conflict markers; TF-IDF initialization/query-vector bugs; BM25 never builds its matrix and inherits TF-IDF failure; the corpus/qrels semantics are still not a proper retrieval benchmark; legacy recall is wrong; full test collection is blocked by dependencies in this environment.

**Not confirmed broken:** the fundamental Transformer encoder equations. The encoder test and vector DB tests pass. The next priority should be data/corpus correctness, lexical baselines, metrics, and repeatable training—not a ground-up Transformer rewrite.

---
*Notebook authored as a review proposal. It does not modify source files or claim full FiQA/SQuAD training validation.*